# Numerical solutions to partial differential equations — Python lab 2
Session 6 · 5CKI2MCP


In [ ]:
# SETUP
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

plt.rcParams.update({"font.size": 11, "axes.grid": True, "grid.alpha": 0.25})



# 1. Sovling the heat equation numerically  on the real line

We want to solve the heat equation
$$
  \frac{\partial u(x,t)}{\partial t} = D \frac{\partial^2 u(x,t)}{\partial x^2}
$$
with the initial condition
$$
  u(x,t=0) = u_0(x)
$$
where $u_0(x)$ is given, $x$ belongs to the whole real line, and $t\ge 0$.
Since we cannot properly take the full real line, we will take a large interval, say $[-L,L]$ an work with periodic bounadry conditions, but we will consider times that are short enough so that we don't practically reach the boundary conditions.


Let's design the numercial scheme.
Let $h$ be the elementary time step and let $a$ be the elemntary spatial interval. Let's first discretize the time derivative, e.g.:
$$
  \frac{\partial u(x,t)}{\partial t} \approx \frac{u(x,t+h) - u(x,t)}{h}.
$$
Therefore, we get the discrete time evolution equation
$$
  u(x,t+h) = u(x,t) + D \frac{\partial^2 u(x,t)}{\partial x^2} h .
$$
Numerically, we still need to approximate the spatial derivative, e.g.:
\begin{align*}
  \frac{\partial^2 u(x,t)}{\partial x^2}
  &\approx
  \frac{d}{dx} \frac{u(x+a,t) - u(x,t)}{a}
  ≈\frac1a
  \left( \frac{u(x+a,t)-u(x,t)}{a} - \frac{u(x,t) - u(x-a,t)}{a} \right)\\
  &=
  \frac{u(x+a,t) - 2 u(x,t) + u(x-a,t)}{a^2}.
\end{align*}


Taking $D=1$ for simplicity (or by changing the units of time), we find the algorithm
$$
  u(x,t+h) = u(x,t) + \frac{h}{a^2} \left(u(x+a,t) - 2 u(x,t) + u(x-a,t)\right)
$$


### Let's implement this explicitly

$T$ is the total time,

$N$ is the number of time steps, so that $h=T/N$,

$L$ is the half-lenght of the interval $[-L,L]$,

$M$ is the number of elementary spatial intervals, so that $a = 2 L/M$,

$u_0$ is the initial profile.

In [ ]:
# Complete the algorithm
def euler_heat(T, N, L, M, u0):
    h = T / N
    a = 2 * L / M

    space = np.linspace(-L, L, M, endpoint=False)
    u = np.zeros((N + 1, M))

    # Initial condition
    for j in range(M):
        u[0, :] = u0

    for n in range(N):
        for j in range(M):

            # Periodic boundary conditions.
            # Recall: k % M is "k modulo M", so in particular M % M = 0 and -1 % M = M-1
            next_u = # TO DO

            u[n + 1, j] = next_u

    return u

### We now apply this to a few initial profiles:

1. A Dirac delta: $\delta (x) = 0$ if $x\ne 0$, $\delta(0)=+\infty$ such that $\int_{-\infty}^{+\infty} \delta(x)dx = 1$. This is not an ordinary function on the real line (there doesn't exist any function with such properties). Once space is discretized, as we do here, it is easy to define: it's zero for $x\ne 0$ and it's $1/a$ when $x=0$.
2. A rectangular pulse:
$$
u_0(x)=
\begin{cases}
1 & \text{if } |x|<0.5,\\
0 & \text{otherwise}.
\end{cases}
$$
3. A triangular pulse:
$$
u_0(x)=\max\left(1-\frac{|x|}{0.5},\,0\right)
$$

4. A spiky thing:
$$
u_0(x)=e^{-(x/0.7)^4}\left(1+0.7\cos(35x)\right)
$$

5. Positive and negative bumps:
$$
u_0(x)=e^{-((x-0.4)/0.2)^2}-e^{-((x+0.4)/0.2)^2}
$$

### What's the common feature of the time evolution?


In [ ]:
#Just run the code: uncomment the initial profile you want to pick

L = 5
M = 200
a = 2 * L / M

T = 0.1
N = 1000

space = np.linspace(-L, L, M, endpoint=False)



# Choose one initial condition (just uncomment the one you want):

# 1. Discrete Dirac delta (total mass = 1)
#u0 = np.zeros(M)
#u0[M // 2] = 1 / a

# 2. Rectangular pulse
#u0 = (np.abs(space) < 0.5).astype(float)

# 3. Triangular pulse
#u0 = np.maximum(1 - np.abs(space) / 0.5, 0)

# 4. A spiky thing
#u0 = np.exp(-(space / 0.7)**4) * (
#    1 + 0.7 * np.cos(35 * space)
#)

# 5. Positive and negative bumps
#u0 = (
#    np.exp(-((space - 0.4) / 0.2)**2)
#    - np.exp(-((space + 0.4) / 0.2)**2)
#)

# Solve the heat equation
u = euler_heat(T, N, L, M, u0)

fig, ax = plt.subplots()
line, = ax.plot(space, u[0, :], lw=2)



ax.set_xlim(-2, 2)
ax.set_ylim(
    np.min(u0) - 0.1 * np.ptp(u0),
    np.max(u0) + 0.1 * np.ptp(u0)
)
ax.set_xlabel("x")
ax.set_ylabel("u(x,t)")

def update(n):
    line.set_ydata(u[n, :])
    ax.set_title(f"t = {n * T / N:.3f}")
    return line,

ani = FuncAnimation(
    fig, update,
    frames=range(0, N + 1, 10),
    interval=50,
    repeat=False
)

plt.close(fig)
display(HTML(ani.to_jshtml()))

# 2. Reaching stationarity with boundary conditions

Let's first imagine that the profile $u(x,t)$ represents the temperature, and let's assume that the temperature is fixed at both ends:
$$
  T(-L,t) = A, \qquad T(L,t) = B
$$
for given $A$ and $B$.

Modify the code so that the value of $u(-L,t)= A$ at all times and $u(L,t)=B$ at all times. In the snapshot below, there are now $M+1$ discretization points on the $x$-axis, indexed by $j=0,1,\dots,M$ and we want the temperature at $j=0$ to be equal to $A$, and the temperature at $j=M$ to be equal to $B$.

In [ ]:
#Complete the algorithm
def euler_heat_fixed(T, N, L, M, u0, A, B):
    h = T / N
    a = 2 * L / M

    u = np.zeros((N + 1, M + 1))
    u[0, :] = u0
    u[0, 0] = A # redundant
    u[0, M] = B # redundant

    for n in range(N):
        for j in range(1, M):
            u[n + 1, j] = # TO DO

        u[n + 1, 0] = # TO DO
        u[n + 1, M] = # TO DO

    return u

Check what happens at long times starting from different initial conditions (3 of them are prpoposed below).

1. What is the stationary profile? Check analytically. Does it depend on the initial condition ?
2. What is the heat flux $J$ in the stationary profile? The heat flux is proportional to the temperature gradient:
$$
  J = - D \frac{\partial u}{\partial x}
$$
where $D$ is the diffusion coefficient (we can set $D=1$). Does $J$ still depends on $x$ and $t$?

In [ ]:
# Just run the code. Uncomment the initial profile that you want.
L = 1
M = 40
a = 2 * L / M

T = 1
N = 1000
h = T / N

space = np.linspace(-L, L, M + 1)
time = np.linspace(0, T, N + 1)

A = 1
B = 0

# Pick an initial profile (uncomment it)

# Example initial profile 1
#u0 = A + (B-A)*(space+L)/(2*L) + 0.8*np.sin(np.pi*(space+L)/(2*L))

# Example initial profile 2
#u0 = (A + (B-A)*(space+L)/(2*L)
#      + 0.5*np.sin(np.pi*(space+L)/L)
#      + 0.3*np.sin(3*np.pi*(space+L)/(2*L)))

# Example initial profile 3
#u0 = (A + (B-A)*(space+L)/(2*L)
#      + 1.5*(1-(space/L)**2)*np.exp(-15*(space/L-0.3)**2))

#u_fixed = euler_heat_fixed(T, N, L, M, u0, A, B)




# visualizing
fig, ax = plt.subplots()

line_fixed, = ax.plot(
    space, u_fixed[0, :], label="Fixed temperatures"
)

ax.set_xlim(-L, L)
margin = 0.1 * (np.max(u0) - np.min(u0))
ax.set_ylim(
    min(np.min(u0), A, B) - margin,
    max(np.max(u0), A, B) + margin)
ax.set_xlabel("x")
ax.set_ylabel("Temperature")
ax.legend()

def update(n):
    line_fixed.set_ydata(u_fixed[n, :])
    ax.set_title(f"t = {time[n]:.2f}")
    return line_fixed

ani = FuncAnimation(
    fig, update,
    frames=range(0, N + 1, 30),
    interval=50,
    repeat=False
)

plt.close(fig)
display(HTML(ani.to_jshtml(default_mode="once")))

# Insulated boundary conditions

Let's now assume that $u(x,t)$ represents heat (rather than temperature). We imaging that the rod is insulated at the boundaries, i.e. there is no heat flux on the left and on the right:
$$
  \frac{\partial u(-L,t)}{\partial x} = \frac{\partial u(L,t)}{\partial x} = 0
$$
at all times.

Here is how it can be implemented in the code. We again consider $M+1$ discretization points on the $x$-axis. We further assume that there is an extra point on the left, $-L-a$, and one extra point on the right, $L+a$, such that
$$
  u(-L-a,t) = u(-L,t) \quad \text{and} \quad
  u(L+a,t) = u(L,t)
$$
at all times $t$. This is a way to implement the boundary conditions.
This implies that the algorithm is modified as follows on the left boundary:
\begin{align*}
  u(-Lt+h) &= u(-L,t) + \frac{h}{a^2}
  \left( u(-L-a,t) - 2 u(-L,t) + u(-L+a,t) \right)\\
  &= u(-L,t) + \frac{h}{a^2} \left( u(-L+a,t) - u(-L,t) \right).
\end{align*}
We do a similar modification on the right boundary.

In [ ]:
# Complete the algorithm
def euler_heat_insulated(T, N, L, M, u0):
    h = T / N
    a = 2 * L / M

    u = np.zeros((N + 1, M + 1))
    u[0, :] = u0

    for n in range(N):
        for j in range(1, M):
            u[n + 1, j] = # TO DO

        u[n + 1, 0] = # TO DO
        )

        u[n + 1, M] = # TO DO

    return u

Check what happens at long times starting from different initial conditions (3 of them are prpoposed below).

1. What is the stationary profile? Check analytically.  Does it depend on the initial condition?
2. Is there a flux in the system?
3. From the physics, what do you think determines the value of the stationary profile?

In [ ]:
# Just run the cell
L = 1
M = 40
a = 2 * L / M

T = 0.8
N = 2000
h = T / N

space = np.linspace(-L, L, M + 1)
time = np.linspace(0, T, N + 1)

# 1. Gaussian centered at the origin
u0_1 = np.exp(-10 * space**2)

# 2. Two bumps of different heights
u0_2 = (
    0.8 * np.exp(-30 * (space + 0.4)**2)
    + 1.5 * np.exp(-30 * (space - 0.3)**2)
)

# 3. A broad asymmetric profile
u0_3 = 0.5 + 0.4 * np.sin(np.pi * space / (2 * L))

u_insulated = euler_heat_insulated(T, N, L, M, u0)


u1 = euler_heat_insulated(T, N, L, M, u0_1)
u2 = euler_heat_insulated(T, N, L, M, u0_2)
u3 = euler_heat_insulated(T, N, L, M, u0_3)

fig, ax = plt.subplots()

line1, = ax.plot(space, u1[0, :], label="Gaussian")
line2, = ax.plot(space, u2[0, :], label="Two bumps")
line3, = ax.plot(space, u3[0, :], label="Asymmetric")

ax.set_xlim(-L, L)
ax.set_ylim(-0.1, 1.7)
ax.set_xlabel("x")
ax.set_ylabel("Temperature")
ax.legend()

def update(n):
    line1.set_ydata(u1[n, :])
    line2.set_ydata(u2[n, :])
    line3.set_ydata(u3[n, :])

    ax.set_title(f"t = {n * h:.2f}")
    return line1, line2, line3

ani = FuncAnimation(
    fig,
    update,
    frames=range(0, N + 1, 30),
    interval=50,
    repeat=False
)

plt.close(fig)
display(HTML(ani.to_jshtml(default_mode="once")))



For each of these three intial profiles, visualize now how the total heat in the system
$$
  Q(t) = \int_{-L}^L u(x,t) dx
$$
evolve with time.

### How is that related to the value of the steady state?

In [ ]:
def total_heat(u, a):
    # np.sum(v, axis=i) will sum the values of the v array along the i direction (i=0 or i=1 for a 2d array)
    return # TO DO

heat1 = total_heat(u1, a)
heat2 = total_heat(u2, a)
heat3 = total_heat(u3, a)

plt.plot(time, heat1, label="Gaussian")
plt.plot(time, heat2, label="Two bumps")
plt.plot(time, heat3, label="Asymmetric")

plt.xlabel("Time")
plt.ylabel("Total heat")
plt.legend()
plt.grid()
plt.show()

# 3. Solving the heat equation with periodic boundary conditions
We already used periodic boundary conditions above, but this was jus because we couldn't take the whole real line on a computer. We now will take seriously that we have a finite interval $[-L,L]$ and assume periodic boundary conditions. So we don't need to change our algorithm.

Here we want to consider a special type of initial profiles:
\begin{align*}
  &u_0(x) = \sin(2\pi k x/2L), \qquad k=1,2,3,\dots, \\
  &u_0(x) = \cos(2\pi k x/2L), \qquad k=0,1,2,\dots
\end{align*}

### (1) Vary $k$ and observe what happens.

### (2) Show that, for all such initial profiles, the solution is of the form
$$
  u(x,t) = f(t) u_0(x)
$$
### where $f(t)$ solves
$$
  \frac{df(t)}{dt} = -\left(\frac{\pi k}{L}\right)^2 f(t), \qquad f(0)=1
$$
### and solves this equation!

In [ ]:
# Just run the code for various values of k
L = 5
M = 200

T = 0.5
N = 1000

space = np.linspace(-L, L, M, endpoint=False)

# Just vary k here
k = 0
u0 = np.cos(np.pi * k * space / L)

# Solve the heat equation
u = euler_heat(T, N, L, M, u0)

fig, ax = plt.subplots()
line, = ax.plot(space, u[0, :], lw=2)



ax.set_xlim(-L, L)
ax.set_ylim(
    np.min(u0) - 0.1 * np.ptp(u0),
    np.max(u0) + 0.1 * np.ptp(u0)
)
ax.set_xlabel("x")
ax.set_ylabel("u(x,t)")

def update(n):
    line.set_ydata(u[n, :])
    ax.set_title(f"t = {n * T / N:.3f}")
    return line,

ani = FuncAnimation(
    fig, update,
    frames=range(0, N + 1, 10),
    interval=50,
    repeat=False
)

plt.close(fig)
display(HTML(ani.to_jshtml()))

## Fourier expansion

We now accept the idea that "every" periodic function $u(x)$ on $[-L,L]$ can be expanded into a series of the above functions:   
$$
  u(x) = \sum_{k=0}^\infty a(k) \cos(2\pi k x/2L) + \sum_{k=1}^\infty b(k) \sin(2\pi k x/2L)
$$
This is called a *Fourier series*. We will see later in this course how to compute the Fourier coefficients $a(k)$ and $b(k)$ for a given function $u(x)$.

Here let's simply check numerically that things work for two functions with given Fourier coefficients. We take $L=1$ for simplicity.

1. A smooth function. Let $u(x)$ be given by
$$
u(x)=\frac{2\cos(\pi x)-1}{5-4\cos(\pi x)}.
$$
We claim that the Fourier coefficients are given by
$$
a(0)=0,\qquad a(k)=\frac1{2^k} \quad (k=1,2,3,...),\qquad b(k)=0.
$$

2. A discontinuous function. Let $u(x)$ be given by
$$
u(x)=
\begin{cases}
1 & \text{if } |x|<\frac12,\\
0 & \text{if } \frac12<|x|\leq1.
\end{cases}
$$
We claim that the Fourier coefficients are given by
$$
a(0)=1,\qquad
a(k)=\frac{2}{k\pi}\sin\left(\frac{k\pi}{2}\right),
\qquad b(k)=0.
$$

### Analyse the convergence of the expansions by varying $K$, which is the number of Fourier coefficients.

In [ ]:
# Just run the code and vary K (the number of terms in the Fourier expansion) as well as the function that you want to approximate
L = 1
M = 500
space = np.linspace(-L, L, M)

K = 50  # Number of Fourier modes

# Choose a function (uncomment it):

# 1. Smooth function
#u = (2*np.cos(np.pi*space) - 1) / (5 - 4*np.cos(np.pi*space))

#def a(k):
#    return 1 / 2**k

#a0 = 0

# 2. Square pulse
#u = (np.abs(space) < 0.5).astype(float)

#def a(k):
#     return 2 * np.sin(k*np.pi/2) / (k*np.pi)

#a0 = 1

# Fourier approximation
approx = np.ones(M) * a0 / 2

for k in range(1, K + 1):
    approx += a(k) * np.cos(k * np.pi * space)  # TO DO

plt.plot(space, u, label="Original function")
plt.plot(space, approx, label=f"Fourier approximation K={K}")
plt.legend()
plt.grid()
plt.show()

## Solving the heat equation with Fourier series

We now take the square pulse initial profile above and we compare two ways of solving the heat equation.

First, we directly apply the numerical method used earlier.

Second, we expand the intial profile in a Fourier series and use the superposition principle: if $u(x,t)$ solves the heat equation for the initial profile $u_0(x)$ and if $v(x,t)$ solves the heat equation for the initial profile $v_0(x)$, then $A u(x,t) + B v(x,t)$ sovles the heat equation for the initial profile $Au_0 (x) + Bv_0(x)$.

So, here, the initial profile is decomposed as
$$
  u_0(x) = \sum_{k=0}^\infty a(k) \cos(2\pi k x/2L) + \sum_{k=1}^\infty b(k) \sin(2\pi k x/2L).
$$
Hence, by a computation that you did earlier, the solution to the heat equation is given by
$$
  u(x,t) = \sum_{k=0}^\infty a(k) e^{-(\pi k/L)^2 t}\cos(2\pi k x/2L) + \sum_{k=1}^\infty b(k) e^{-(\pi k/L)^2 t} \sin(2\pi k x/2L).
$$


In [ ]:
# Complete the code, using the solution displayed above
L = 1
M = 100
a = 2 * L / M

T = 0.2
N = 2000
h = T / N

space = np.linspace(-L, L, M, endpoint=False)

# Square pulse initial condition
u0 = (np.abs(space) < 0.5).astype(float)

# Numerical solution
u = euler_heat(T, N, L, M, u0)

# Fourier solution
K = 20

def fourier_heat(t):
    result = np.ones(M) * 0.5

    for k in range(1, K + 1):
        coefficient = 2 * np.sin(k * np.pi / 2) / (k * np.pi)

        result += # TO DO

    return result

# Animated comparison
fig, ax = plt.subplots()

line_euler, = ax.plot(space, u[0, :], label="Euler", lw=2)
line_fourier, = ax.plot(
    space, fourier_heat(0), "--", label="Fourier", lw=2
)

ax.set_xlim(-L, L)
ax.set_ylim(-0.2, 1.2)
ax.set_xlabel("x")
ax.set_ylabel("u(x,t)")
ax.legend()

def update(n):
    t = n * h

    line_euler.set_ydata(u[n, :])
    line_fourier.set_ydata(fourier_heat(t))

    ax.set_title(f"t = {t:.4f}")
    return line_euler, line_fourier

ani = FuncAnimation(
    fig,
    update,
    frames=range(0, N + 1, 20),
    interval=50,
    repeat=False
)

plt.close(fig)
display(HTML(ani.to_jshtml(default_mode="once")))